# GNN edge removal + GAT / GAT-sep (v5-gnn-2026-10-01)
**Before running:** Session options → **Accelerator = GPU T4 x2**, **Internet = On**; Input → attach the dataset containing **`gnn_edge_removal_code_v5.zip`**.
Step 1: set `MODE = 'validate'` and run all cells by hand (~10 min). It checks that GCN/SAGE reproduce the paper.
Step 2: set `MODE = 'full'` → **Save Version → Save & Run All**. Download `results_gnn_<mode>.zip` from Output.
**v5:** set `MODE = 'v5'` → **Save Version → Save & Run All** (about 3–4 h). It adds GCN-res, the MLP baseline, all 10 Roman-empire splits, Amazon-ratings and Tolokers. The v4 results ship in `prior/` and are reused, not rerun.

In [ ]:
# 1. Copy the code from the attached dataset
import glob, os, shutil, zipfile
EXPECTED = 'v5-gnn-2026-10-01'
W = '/kaggle/working/gnn'
for z in glob.glob('/kaggle/input/**/gnn_edge_removal_code_v5.zip', recursive=True):
    zipfile.ZipFile(z).extractall('/kaggle/tmp_gnn')
versions = glob.glob('/kaggle/input/**/CODE_VERSION.txt', recursive=True) + glob.glob('/kaggle/tmp_gnn/**/CODE_VERSION.txt', recursive=True)
found = {v: open(v).read().strip() for v in versions}
print('code versions found:', found)
good = [v for v, ver in found.items() if ver == EXPECTED]
assert good, f'Attach the dataset with gnn_edge_removal_code_v5.zip (version {EXPECTED}). Found: {found}'
shutil.copytree(os.path.dirname(good[0]), W, dirs_exist_ok=True)
os.chdir(W)
print('using code from:', os.path.dirname(good[0]))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install PyTorch Geometric (keeps Kaggle's GPU PyTorch) and download every dataset once
!pip install -q torch_geometric
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on Accelerator = GPU T4 x2'
from run import load
for name in ('cora', 'roman', 'amazon', 'tolokers'):
    d, tr, va, te = load(name, 'data')
    print(name, d, '| train/val/test', int(tr.sum()), int(va.sum()), int(te.sum()))

In [ ]:
# 3. Checks: unit tests + a forward pass through all 6 models
!python -m pytest -q tests
!python models.py

In [ ]:
# 4. Run.  'validate' = Cora only, GCN + SAGE at the paper's operating points, 5 seeds, input dropout OFF,
#                        with Cora feature normalisation ON (GPU 0) and OFF (GPU 1).
#           'full'     = all 4 models x 3 strategies x 20 sparsity levels x 5 seeds (2,400 runs) on both GPUs,
#                        using the settings chosen by validation.
#           'v5'       = GCN-res + MLP on Cora and Roman split 0 (5 seeds), then all 6 models on Roman-empire,
#                        Amazon-ratings and Tolokers, splits 0-9 with seed = split. v4 cells are skipped.
MODE = 'v5'
INPUT_DROPOUT = 0        # settled by v2 validation: Roman-empire reproduces within 0.4 points with input dropout OFF
CORA_NORMALIZE = 0       # settled by v3 validation: raw Cora features reproduce the paper within 1.1 points

import subprocess, time
os.makedirs('results', exist_ok=True)
if MODE == 'validate':
    v = ['--mode', 'validate', '--datasets', 'cora', '--models', 'gcn', 'sage', '--input-dropout', '0']
    jobs = {0: v + ['--cora-normalize', '1', '--out', 'results/validate_coranorm1.csv'],
            1: v + ['--cora-normalize', '0', '--out', 'results/validate_coranorm0.csv']}
else:
    flags = ['--input-dropout', str(INPUT_DROPOUT), '--cora-normalize', str(CORA_NORMALIZE)]
    jobs = {0: ['--datasets', 'cora', 'roman', '--models', 'gcn', 'sage', *flags, '--out', 'results/part0.csv'],
            1: ['--datasets', 'roman', 'cora', '--models', 'gat', 'gatsep', *flags, '--out', 'results/part1.csv']}
if MODE == 'v5':
    flags = ['--input-dropout', str(INPUT_DROPOUT), '--cora-normalize', str(CORA_NORMALIZE),
             '--done-from', 'prior/part0.csv', 'prior/part1.csv']
    multi = ['--datasets', 'roman', 'amazon', 'tolokers', '--splits', *map(str, range(10)), '--paired', *flags]
    # GPU 0: the cheap models (incl. the 5-seed split-0 runs); GPU 1: the two attention models
    jobs = {0: [['--datasets', 'cora', 'roman', '--models', 'gcnres', 'mlp', *flags, '--out', 'results/v5_gpu0.csv'],
                [*multi, '--models', 'gcn', 'sage', 'gcnres', 'mlp', '--out', 'results/v5_gpu0.csv']],
            1: [[*multi, '--models', 'gat', 'gatsep', '--out', 'results/v5_gpu1.csv']]}
else:
    jobs = {g: [a] for g, a in jobs.items()}
procs, logs = {}, {}
for gpu, steps in jobs.items():
    logs[gpu] = f'results/gpu{gpu}_{MODE}.log'
    cmd = ' && '.join('python run.py ' + ' '.join(a) for a in steps)
    procs[gpu] = subprocess.Popen(cmd, shell=True, stdout=open(logs[gpu], 'w'), stderr=subprocess.STDOUT,
                                  env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
seen = {g: 0 for g in jobs}
while True:
    for g in jobs:
        lines = open(logs[g]).read().splitlines()
        for line in lines[seen[g]:]:
            if 'Warning' not in line:
                print(f'[gpu{g}] {line}', flush=True)
        seen[g] = len(lines)
    if all(p.poll() is not None for p in procs.values()):
        break
    time.sleep(30)
assert all(p.returncode == 0 for p in procs.values()), 'A run failed - read the [gpu] lines above'
print('done')

In [ ]:
# 5. Analysis + results zip (download from the Output tab)
if MODE == 'validate':
    for cn in (1, 0):
        print(f'================ CORA NORMALISE = {cn} (input dropout off, 5 seeds) ================')
        !python analyze.py results/validate_coranorm{cn}.csv --metric test_final --outdir results/val_coranorm{cn} 2>&1 | grep -v -i warn | sed -n '1,10p'
elif MODE == 'v5':
    !python analyze_v5.py prior/part0.csv prior/part1.csv results/v5_gpu0.csv results/v5_gpu1.csv --outdir results/v5 2>&1 | grep -v -i warn
else:
    !python analyze.py results/part0.csv results/part1.csv --metric test_final --outdir results 2>&1 | grep -v -i warn
!cd /kaggle/working && zip -qr results_gnn_{MODE}.zip gnn/results && ls -lh results_gnn_{MODE}.zip